In [0]:
%sql
create connection if not exists test_earthquake_conn type HTTP OPTIONS (
  host = "https://earthquake.usgs.gov",
  port = 443,
  base_path = '/earthquakes/feed/v1.0/',
  bearer_token = 'na'
)

In [0]:
%python
from databricks.sdk import WorkspaceClient
w = WorkspaceClient()

conn = w.connections.get("test_earthquake_conn")
print(conn)
base_url=f"{conn.options['host']}{conn.options['base_path']}"


In [0]:
%python
dbutils.widgets.text("catalog_name","test_dev")
catalog_name = dbutils.widgets.get("catalog_name")

In [0]:
%python
spark.sql(f"use catalog {catalog_name}");
spark.sql("use schema bronze");
spark.sql("create volume if not exists earthquake_data");


In [0]:
%python
import requests
import json
import datetime

url = f'{base_url}summary/all_day.geojson'
response = requests.get(url)
if (response.status_code != 200):
  raise Exception(f"Error: {response.status_code} ")
data=response.json()
current_date = datetime.datetime.now().strftime("%Y-%m-%d")
       
#
dbutils.fs.put(f"/Volumes/{catalog_name}/bronze/earthquake_data/earthquake_data_{current_date}.json", json.dumps(data), overwrite=True)
       